# DMPJ 自己対戦RL（Colab 用）

このノートブックは DMPJ の自己対戦学習を Colab で大きく回すためのものです。
実行すると、リポジトリを取得し、自己対戦でモデルを学習し、学習前後の勝率
（対ランダム／対greedy）を表示します。結果（model.npz とログ）は最後のセルで
Google Drive に保存するか、ダウンロードできます。それを渡してもらえれば、
続きの検証と学習をこちらで行えます。

注意: 収集（対戦の実行）は CPU が律速なので、GPU ランタイムでなくても構いません。
torch が使える場合は将来の torch 版モデルの学習に使えます。

In [ ]:
# リポジトリ取得（Phase 10 ブランチ: torch 版モデル）
!git clone -q -b work/phase10-torch https://github.com/KOBA1616/DMPJ.git
%cd DMPJ
!pip install -q pytest 2>/dev/null | tail -1

In [ ]:
# 環境確認（torch があればバージョンを表示）
import sys, numpy
print(sys.version)
print('numpy', numpy.__version__)
try:
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
except ImportError:
    print('torch なし（numpy モデルで学習します）')

In [ ]:
# torch 版モデルの検証（numpy 版との loss/勾配パリティ、数分）
!python -m pytest tests/test_ai_torch.py -q 2>&1 | tail -2

In [ ]:
# 学習の実行（torch 版モデル。規模はここで調整）
# 本番規模の目安: 200 iter × 16局 = 3200局
!DMSIM_RUST=0 python tools/selfplay_train.py --model torch --deck data/decks/saga.yaml --iters 200 --games-per-iter 16 --eval-games 40 --opponents random,greedy --workers 2 --out /content/model_colab_torch.npz

In [ ]:
# モデルのダウンロード（スクリプトからブラウザのダウンロードを開始する）
from google.colab import files
files.download('/content/model_colab_torch.npz')
# うまくいかない場合の代替: Drive 保存
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/model_colab_torch.npz /content/drive/MyDrive/